# Image → 3D Scene (Stage B · Trellis)

Upload one photo → get a `scene.glb` with **real 3D generated assets** (not flat cut-outs), each detected object reconstructed by Trellis and placed on a fitted ground plane.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Run the cells top to bottom. The install cell takes ~15–25 min the first time (Trellis compiles CUDA ops) and is the part most likely to need a retry — if it errors, re-run it; if it still fails, send me the output.

In [ ]:
# 0. Confirm we have a GPU
!nvidia-smi -L

## 1. Install Trellis + dependencies (one-time per session)

In [ ]:
import os
os.environ['ATTN_BACKEND'] = 'xformers'   # 'flash-attn' also works
os.environ['SPCONV_ALGO'] = 'native'      # avoids a slow first-run autotune

# Clone Trellis and run its installer (compiles the custom CUDA ops).
![ -d /content/TRELLIS ] || git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
%cd /content/TRELLIS
!. ./setup.sh --basic --xformers --flash-attn --diffoctreerast --spconv --mipgaussian --kaolin --nvdiffrast

# App-side deps used by our detection / depth / geometry code.
!pip install -q "transformers>=4.44,<5" timm accelerate huggingface_hub trimesh xatlas scipy

## 2. Get the pipeline code

In [ ]:
import sys
REPO = '/content/image-3d-pipeline'
BRANCH = 'claude/sweet-cori-kzkhyr'
![ -d {REPO} ] || git clone --branch {BRANCH} https://github.com/sanjanamani/image-3d-pipeline.git {REPO}
!cd {REPO} && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('pipeline code ready:', REPO)

## 3. Upload your photo

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = '/content/' + next(iter(uploaded))
print('uploaded:', IMAGE_PATH)

## 4. Build the 3D scene

Detects objects → estimates depth → runs Trellis per object → places each on the fitted ground. Several minutes per object on a T4.

In [ ]:
import os
os.chdir(REPO)
from scene_build import run_build
glb_path = run_build(IMAGE_PATH, output_dir='/content/outputs')
print('\nDONE ->', glb_path)

## 5. View it inline + download

In [ ]:
import base64
from IPython.display import HTML, display

b64 = base64.b64encode(open(glb_path, 'rb').read()).decode()
display(HTML(f'''
<script type="module" src="https://unpkg.com/@google/model-viewer/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}"
  camera-controls auto-rotate shadow-intensity="1"
  style="width:100%;height:520px;background:#222;"></model-viewer>
'''))

from google.colab import files as _f
_f.download(glb_path)